<div style="
background:#121212;
padding:35px;
border:3px solid #d4af37;
border-radius:15px;
margin:20px 0;
">

<h1 style="
font-size:35px;
color:#d4af37;
text-align:center;
margin-top:0;
margin-bottom:10px;
">
Export Data
</h1>

<h2 style="
font-size:25px;
color:#ffffff;
text-align:center;
font-weight:normal;
margin-top:0;
margin-bottom:25px;
">
Análisis del comportamiento de los saldos bancarios durante 2025
</h2>

<hr style="border: 5px solid #d4af37; margin-bottom:30px;">
<hr style="border: 3px solid #d4af37; margin-bottom: 30px;">
<hr style="border: 1px solid #d4af37; margin-bottom: 15px;">
</div>


In [1]:
# Librerias

import os
import re
import pickle
from math import pi
from pathlib import Path
import nbformat
import numpy as np
import pandas as pd
from bokeh.io import export_png, output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, CustomJS, DatetimeTickFormatter, GlobalInlineStyleSheet, HoverTool, Label, Legend, LegendItem, NumeralTickFormatter, RangeTool, TapTool
from bokeh.plotting import figure, show
from bokeh.transform import dodge, factor_cmap
from bs4 import BeautifulSoup
from IPython.display import HTML, display

In [2]:
# Function Dataframes 

def guardar_tabla(
    df: pd.DataFrame, nombre: str, output_dir: str = "tables"
) -> None:
  """Toma un DataFrame de Pandas, aplica el formato de vista previa tipo Jupyter

  (si es muy grande) o completo (si es pequeño), y exporta un archivo .tex
  con formato profesional usando booktabs.
  """
  os.makedirs(output_dir, exist_ok=True)

  total_filas, total_cols = df.shape
  limite_visualizacion = 10 

  if total_filas > limite_visualizacion:
    puntos = pd.DataFrame(
        [["..."] * total_cols], columns=df.columns, index=["..."]
    )
    df_mostrar = pd.concat([df.head(5), puntos, df.tail(5)])
  else:
    df_mostrar = df.copy()

  latex_str = df_mostrar.to_latex(
      index=True,
      escape=True,
      column_format="l" + "c" * total_cols,
      float_format="%.2f",
  )

  if total_filas > limite_visualizacion:
    pie_nota = (
        f"\\smallskip\\textit{{Mostrando las primeras 5 y últimas 5 filas"
        f" de un total de {total_filas:,} registros $\\times$ {total_cols}"
        " variables.}}"
    )
  else:
    pie_nota = (
        f"\\smallskip\\textit{{Conjunto completo de datos: {total_filas:,}"
        f" registros $\\times$ {total_cols} variables.}}"
    )

  contenido_final = f"""% Archivo generado automáticamente. No editar a mano.
\\begin{{table}}[htbp]
\\centering
\\begin{{adjustbox}}{{max width=\\textwidth}}
{latex_str}
\\end{{adjustbox}}
\\caption{{Resultado de la variable: \\texttt{{{nombre}}}}}
\\label{{tab:{nombre}}}
{pie_nota}
\\end{{table}}
"""
  ruta_archivo = os.path.join(output_dir, f"{nombre}.tex")
  with open(ruta_archivo, "w", encoding="utf-8") as f:
    f.write(contenido_final)

  print(
      f"-> Tabla '{nombre}.tex' exportada con éxito en la carpeta"
      f" '{output_dir}/'."
  )


In [3]:
from bokeh.io.export import export_png
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


def guardar_grafica(figura, nombre, output_dir="../charts"):

    os.makedirs(output_dir, exist_ok=True)

    ruta_archivo = os.path.join(
        output_dir,
        f"{nombre}.png"
    )

    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(
            ChromeDriverManager().install()
        ),
        options=options
    )

    try:
        export_png(
            figura,
            filename=ruta_archivo,
            webdriver=driver
        )

        print(
            f"Gráfica guardada: {ruta_archivo}"
        )

    finally:
        driver.quit()

In [5]:
# Function Text

def extraer_textos_notebook(ruta_notebook):
    """
    Extrae las secciones marcadas con:
    <!-- TIPO: XXXXX -->

    Retorna un diccionario con todo el contenido.
    """

    with open(ruta_notebook, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    reporte = {}
    for celda in nb.cells:
        if celda.cell_type != "markdown":
            continue

        contenido = celda.source
        patron = r"<!--\s*EXPORT:\s*(.*?)\s*-->"
        encontrado = re.search(patron, contenido)

        if not encontrado:
            continue

        tipo = encontrado.group(1).strip().lower()
        soup = BeautifulSoup(contenido, "html.parser")
        texto = soup.get_text(separator="\n", strip=True)
        reporte.setdefault(tipo, []).append(texto)

    return reporte

def guardar_textos_tex(reporte, output_dir="../text"):
    """
    Guarda los textos extraídos del notebook en archivos .tex.
    """
    # Crear la carpeta si no existe
    os.makedirs(output_dir, exist_ok=True)
    for tipo, bloques in reporte.items():

        for i, texto in enumerate(bloques, start=1):
            nombre = tipo

            if len(bloques) > 1:
                nombre += f"_{i:02d}"   # 01, 02, 03...

            ruta_archivo = os.path.join(output_dir, f"{nombre}.tex")
            with open(ruta_archivo, "w", encoding="utf-8") as f:
                f.write(texto)

            print(f"-> '{nombre}.tex' guardado en '{output_dir}/'")

def exportar_reporte_textos(ruta_notebook):

    reporte = extraer_textos_notebook(ruta_notebook)
    guardar_textos_tex(reporte)

    print("Textos exportados correctamente.")

In [14]:
import pickle

with open("../reporte_estado.pkl", "rb") as f:
    estado = pickle.load(f)

TABLES = estado["tables"]

print("Estado cargado correctamente")
print(TABLES.keys())

Estado cargado correctamente
dict_keys(['table', 'table_ques1', 'table_ques2', 'table_ques3', 'table_ques4', 'table_ques5'])


In [15]:
# Save Table

for nombre, df in TABLES.items():
    guardar_tabla(
        df=df,
        nombre=nombre,
        output_dir="../tables"
    )

print("Todas las tablas exportadas correctamente")

-> Tabla 'table.tex' exportada con éxito en la carpeta '../tables/'.
-> Tabla 'table_ques1.tex' exportada con éxito en la carpeta '../tables/'.
-> Tabla 'table_ques2.tex' exportada con éxito en la carpeta '../tables/'.
-> Tabla 'table_ques3.tex' exportada con éxito en la carpeta '../tables/'.
-> Tabla 'table_ques4.tex' exportada con éxito en la carpeta '../tables/'.
-> Tabla 'table_ques5.tex' exportada con éxito en la carpeta '../tables/'.
Todas las tablas exportadas correctamente


In [16]:
import pickle

with open("../reporte_estado.pkl", "rb") as f:
    estado = pickle.load(f)

print(estado.keys())
print(estado["charts"])
import os

ruta_charts = "../charts"

print(os.listdir(ruta_charts))

dict_keys(['tables', 'charts'])
['chart_ques1', 'chart_ques2', 'chart_ques3', 'chart_ques4', 'chart_ques5']
[]


In [18]:
exportar_reporte_textos("Caso 1.ipynb")

-> 'titulo.tex' guardado en '../text/'
-> 'contexto.tex' guardado en '../text/'
-> 'objetivo.tex' guardado en '../text/'
-> 'pregunta_1.tex' guardado en '../text/'
-> 'analisis_1.tex' guardado en '../text/'
-> 'pregunta_2.tex' guardado en '../text/'
-> 'analisis_2.tex' guardado en '../text/'
-> 'pregunta_3.tex' guardado en '../text/'
-> 'analisis_3.tex' guardado en '../text/'
-> 'pregunta_4.tex' guardado en '../text/'
-> 'analisis_4.tex' guardado en '../text/'
-> 'pregumta_5.tex' guardado en '../text/'
-> 'analisis_5.tex' guardado en '../text/'
-> 'conclusiones.tex' guardado en '../text/'
-> 'recomemdaciones.tex' guardado en '../text/'
Textos exportados correctamente.
